Starting Reinforced Learning

In [ ]:
# code from AI to learn from
import gymnasium as gym
import numpy as np
import random

# 1. Create the 2D game environment using Gymnasium
# "is_slippery=False" makes it easier for the AI to learn at first
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")

# 2. Setup the AI's memory (The Q-Table Spreadsheet)
# Rows = 16 grid squares, Columns = 4 directions (Left, Down, Right, Up)
state_size = env.observation_space.n
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))

# Learning settings (Hyperparameters)
learning_rate = 0.8
discount_factor = 0.95
exploration_rate = 1.0  # Starts out 100% random to explore the map
exploration_decay = 0.995


# 3. Dynamic Training Loop
for episode in range(1000):
    state, info = env.reset()
    done = False
    
    while not done:
        # Exploration vs Exploitation choice
        # If a random number is less than exploration_rate, guess randomly.
        # Otherwise, look at the spreadsheet and pick the best known move!
        if random.uniform(0, 1) < exploration_rate:
            action = env.action_space.sample() # Random guess
        else:
            action = np.argmax(q_table[state, :]) # Smart choice from spreadsheet
            
        # Take the action in the Gymnasium 2D game
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # THE DYNAMIC REINFORCEMENT MATH:
        # Update the spreadsheet cell based on the reward received
        best_future_q = np.max(q_table[next_state, :])
        q_table[state, action] = q_table[state, action] + learning_rate * (reward + discount_factor * best_future_q - q_table[state, action])
        
        state = next_state
        
    # Slowly stop guessing randomly as the spreadsheet gets smarter
    exploration_rate *= exploration_decay

print("✅ Training Complete!")


✅ Training Complete!


In [9]:
import time

# We build a fresh environment with "human" mode to open the Pygame window
visual_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")

state, info = visual_env.reset()
done = False

# The AI plays 1 round using its completed spreadsheet memory
while not done:
    # Always pick the absolute best move from the learned Q-table
    action = np.argmax(q_table[state, :]) 
    
    state, reward, terminated, truncated, info = visual_env.step(action)
    done = terminated or truncated
    
    # Pause for 0.5 seconds per step so human eyes can keep up with the window!
    time.sleep(0.5) 
visual_env.close() # Closes the popup window safely

# My breakdown

**Gymnasium** 
using this library I am able to create game enviroments for the AI to play around in allowing me to train the model using Reinforced Learning (RL) with ease


**Creating the Env**
using the gym library using this line of code I am able to create the enviroment needed
```python
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
```
env: This is the Variable the game will be stored in\
param 1: The game name\
param 2: this game has the option to make the floor slippery / ice\
param 3: this is the way we see the game in output ansi is text and human is the game itself

```python
state_size = env.observation_space.n
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))
```

this is what the "brain" of the model is. the state_size is the grids in our 2d game with there being 16 as its a\
4 x 4 grid leading to 16 possible locations \

the action size is the amounts of moves the ai model can make with there being the 4 cardinal directions\

and finally the q_table where we store the values of each grid that is updated during training to show which moves are the best to take



**Main Loop**

First we reset the game and loop through a game state. using the exploration rate wich starts first at 1 or fully random then closer to \
0 where it goes based off the q_table it will decide where to move with the first option being fully random then the second option \
being the best possible move based on the q_table

```python
        if random.uniform(0, 1) < exploration_rate:
            action = env.action_space.sample() # Random guess
        else:
            action = np.argmax(q_table[state, :]) # Smart choice from spreadsheet
```

```python
        # Take the action in the Gymnasium 2D game
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # THE DYNAMIC REINFORCEMENT MATH:
        # Update the spreadsheet cell based on the reward received
        best_future_q = np.max(q_table[next_state, :])
        q_table[state, action] = q_table[state, action] + learning_rate * (reward + discount_factor * best_future_q - q_table[state, action])
        
        state = next_state
```

Then the model will use the action it got fro either being random or from the q_table and take that step which will give us some details such as\
next_state or the next grid we will be on, the reward for going there, if the game ended, and other info \
with this we see if we terminited if so then we are done if not continue for more cycles. then we find the best next move and after using the math above we update the values inside of the grid




In [33]:
import gymnasium as gym
import numpy as np
import random
import time

# ==========================================
# SETUP & THE DISCRETIZATION (ROUNDING) ENGINE
# ==========================================
NUM_BUCKETS = (10, 10, 20, 20) 

STATE_BOUNDS = [
    [-4.8, 4.8],      
    [-3.0, 3.0],      
    [-0.418, 0.418],  
    [-3.5, 3.5]       
]

def get_discrete_state(continuous_state):
    """Converts raw continuous physics decimals into a tuple of bucket integers."""
    bucket_indices = []
    for i in range(len(continuous_state)):
        boundaries = np.linspace(STATE_BOUNDS[i][0], STATE_BOUNDS[i][1], NUM_BUCKETS[i] - 1)
        bucket = int(np.digitize(continuous_state[i], boundaries))
        bucket_indices.append(bucket)
    return tuple(bucket_indices)


# ==========================================
# PHASE 1: STREAK-BASED TRAINING (BLIND MODE)
# ==========================================
train_env = gym.make("CartPole-v1", render_mode=None)
action_size = train_env.action_space.n

q_table = np.zeros(NUM_BUCKETS + (action_size,))
best_q_table = np.zeros(NUM_BUCKETS + (action_size,))
best_score = 0
consecutive_wins = 0  # Tracks our win streak!

learning_rate = 0.1
discount_factor = 0.99
exploration_rate = 1.0  
exploration_decay = 0.9995

print("🏋️ Training the CartPole AI to be a True Master...")
start_time = time.time()

# Increased maximum episodes since we are waiting for a strict 10-win streak
for episode in range(25000):
    raw_state, info = train_env.reset()
    discrete_state = get_discrete_state(raw_state)
    done = False
    total_frames = 0
    
    while not done:
        if random.uniform(0, 1) < exploration_rate:
            action = train_env.action_space.sample() 
        else:
            action = np.argmax(q_table[discrete_state])

        next_raw_state, reward, terminated, truncated, info = train_env.step(action)
        done = terminated or truncated
        total_frames += 1
        
        if terminated and total_frames < 450:
            reward = -20
            
        next_discrete_state = get_discrete_state(next_raw_state)
        best_future_q = np.max(q_table[next_discrete_state])
        
        current_cell = discrete_state + (action,)
        q_table[current_cell] = q_table[current_cell] + learning_rate * (reward + discount_factor * best_future_q - q_table[current_cell])
        
        discrete_state = next_discrete_state
        
    # --- UPGRADED STREAK CHECKER ---
    if total_frames >= 500:
        consecutive_wins += 1
    else:
        consecutive_wins = 0  # Break the streak if it drops early!
        
    if total_frames > best_score:
        best_score = total_frames
        best_q_table = np.copy(q_table) 
        print(f"🔥 New High Score Saved: {best_score} frames (Episode {episode})")
        
    # Early stop ONLY if it wins 10 distinct rounds in a row
    if consecutive_wins >= 10:
        print(f"\n🎯 AI Mastered! Hit a perfect score of 500 for 10 consecutive rounds early on episode {episode}!")
        break
        
    exploration_rate *= exploration_decay

train_env.close()
print(f"✅ Training Complete! Took {time.time() - start_time:.2f} seconds.")


# ==========================================
# PHASE 2: EVALUATION (SHOW OFF WINDOW)
# ==========================================
print("\n📺 Opening Pygame window to test the ultimate master model...")
visual_env = gym.make("CartPole-v1", render_mode="human")

raw_state, info = visual_env.reset()
discrete_state = get_discrete_state(raw_state)
done = False
eval_frames = 0

while not done:
    action = np.argmax(best_q_table[discrete_state])
    next_raw_state, reward, terminated, truncated, info = visual_env.step(action)
    done = terminated or truncated
    discrete_state = get_discrete_state(next_raw_state)
    eval_frames += 1
    time.sleep(0.02) 

print(f"🏁 Showcase finished! The saved model balanced the pole for {eval_frames} frames.")
visual_env.close()

🏋️ Training the CartPole AI to be a True Master...
🔥 New High Score Saved: 20 frames (Episode 0)
🔥 New High Score Saved: 26 frames (Episode 3)
🔥 New High Score Saved: 30 frames (Episode 11)
🔥 New High Score Saved: 31 frames (Episode 12)
🔥 New High Score Saved: 38 frames (Episode 23)
🔥 New High Score Saved: 61 frames (Episode 36)
🔥 New High Score Saved: 78 frames (Episode 39)
🔥 New High Score Saved: 82 frames (Episode 115)
🔥 New High Score Saved: 84 frames (Episode 322)
🔥 New High Score Saved: 105 frames (Episode 531)
🔥 New High Score Saved: 128 frames (Episode 565)
🔥 New High Score Saved: 139 frames (Episode 802)
🔥 New High Score Saved: 160 frames (Episode 854)
🔥 New High Score Saved: 172 frames (Episode 1003)
🔥 New High Score Saved: 176 frames (Episode 1111)
🔥 New High Score Saved: 229 frames (Episode 1179)
🔥 New High Score Saved: 230 frames (Episode 1219)
🔥 New High Score Saved: 235 frames (Episode 1240)
🔥 New High Score Saved: 259 frames (Episode 1321)
🔥 New High Score Saved: 315 fr